# Chapter 9 — Environment Bugs

**Book alignment:** Debugging AI From First Principles, Chapter 9

**Question this notebook isolates:** Same code hash, same input hash — passes on the laptop,
`KeyError` in CI. Does the locked-container probe separate **H1** (dirty tree / stale cache
masking a real defect) from **H2** (genuine dependency drift), and does a freeze-diff
bisect then name the one package responsible?

In [ ]:
import hashlib, json

def sha(obj):
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:8]

# a CSV-ish loader whose "" -> NaN behaviour changed between pandas 1.5 and 2.x
def load_amounts(rows, *, pandas_major=1):
    out = []
    for r in rows:
        r = dict(r)
        if pandas_major >= 2 and r.get("refund_id", "MISSING") == "":
            r.pop("refund_id")                      # 2.x parses "" as NaN -> key effectively absent
        out.append(r)
    return out

def report(rows, *, pandas_major=1, code_defect=False):
    rows = load_amounts(rows, pandas_major=pandas_major)
    total = 0.0
    for r in rows:
        # a "latent" code defect that a warm cache on the laptop happens to mask
        key = "refund_id" if not code_defect else "refund_ref"
        total += 100.0 - (5.0 if r[key] else 0.0)
    return round(total, 2)

INPUT = [{"order_id": 1, "refund_id": "RB-1"}, {"order_id": 2, "refund_id": ""}]

## 1. Identical code hash + identical data hash, different outcome

In [ ]:
code_hash = "abc1234"                 # pretend git rev-parse matches on both sides
data_hash = sha(INPUT)
print(f"code {code_hash}  data {data_hash}  (identical laptop and CI)")

def outcome(pandas_major, code_defect=False):
    try:
        return report(INPUT, pandas_major=pandas_major, code_defect=code_defect)
    except KeyError as e:
        return f"KeyError {e}"

print("laptop (pandas 1.5):", outcome(1))
print("CI     (pandas 2.1):", outcome(2))
assert outcome(1) == 195.0 and str(outcome(2)).startswith("KeyError")
print("\nsame code, same data -> the divergence lives underneath both")

## 2. The locked-container probe: H1 (masking) vs H2 (drift)

Run the pinned repro in a fresh env built from the lockfile — no caches, no local packages.

In [ ]:
# H2 prediction: both locked runs AGREE with each other once the substrate is pinned.
# H1 prediction: the locked run FAILS EVERYWHERE (the laptop pass was a stale-env artifact).
locked_v2_a = outcome(2)
locked_v2_b = outcome(2)
assert locked_v2_a == locked_v2_b                    # deterministic under a pinned substrate
print(f"locked container (pinned 2.1), run x2: {locked_v2_a} == {locked_v2_b}")

# and pinning BACK to 1.5 makes the side-to-side difference vanish:
assert outcome(1) == 195.0
print("side-to-side difference disappears when the substrate is identical -> H2 (environment drift)")

## 3. Bisect the freeze diff — one entry per run until the outcome flips

In [ ]:
freeze_laptop = {"pandas": "1.5.3", "numpy": "1.26.4", "pyarrow": "14.0.1"}
freeze_ci     = {"pandas": "2.1.0",  "numpy": "1.26.4", "pyarrow": "14.0.1"}
suspects = [k for k in freeze_ci if freeze_ci[k] != freeze_laptop[k]]
print("freeze diff:", {k: (freeze_laptop[k], freeze_ci[k]) for k in suspects})

# flip one entry: downgrade pandas in the CI env -> does the outcome flip?
convicted = None
for pkg in suspects:
    major = 1 if pkg == "pandas" else 2        # simulate the downgrade
    if outcome(major) == 195.0:
        convicted = pkg
        break
assert convicted == "pandas"
print(f"\nconvicted: {convicted} 1.5.3 -> 2.1.0  (CSV dtype inference changed)")
print("fix = lockfile pin + a keep_default_na=False / dtype= guard at the loader (a Ch8 contract)")

## 4. Unseeded randomness and clock mimic drift perfectly

In [ ]:
import random

def flaky(seed=None):
    r = random.Random(seed)
    return sum(r.random() for _ in range(3))

print("unseeded:", round(flaky(), 4), "vs", round(flaky(), 4), " (looks like drift)")
assert flaky(0) == flaky(0)          # pinned seed -> identical
print("pinned seed:", flaky(0), "==", flaky(0))
print("pin seed + PYTHONHASHSEED=0 + TZ=UTC BEFORE claiming environment drift")

## What we earned

Identical code hash and data hash with different outcomes is the signal that the divergence
lives *below* both. The locked-container probe distinguishes a stale-env artifact (H1 —
fails everywhere once pinned) from genuine drift (H2 — the side-to-side difference vanishes
once the substrate is identical), and a one-entry-per-run bisect of the `pip freeze` diff
names `pandas 1.5 → 2.x`. Unseeded RNG and an unpinned clock impersonate drift — exclude
them first.

**Notebook 10 / Chapter 10** opens Part III with a program whose *displayed* form differs
from the one the kernel actually ran: the notebook.